# GitHub作業用 Colab ノートブック
GitHubから導入する重いツール（音声解析・高解像度化・TTSなど）を、ColabのGPUで試すための作業ノート。

**最初に必ず**：メニュー「ランタイム」→「ランタイムのタイプを変更」→ ハードウェア アクセラレータで **T4 GPU** を選んで保存。

**使い分けのルール**
- 重い・単発の処理 → このColab（GPU）
- 毎晩の自動制作で無人で使うもの → ローカル環境（従来どおり）
- Googleアカウントに上げる素材は**健全なものだけ**。NSFW寄りの素材はローカルで処理する。

## ① ファイルのアップロード
処理したい音声（wav/mp3）や画像をアップロードする。アップロードしたファイル名に合わせて、下のセル内の `input.wav` / `input.png` の名前を書き換える。

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

## ② WhisperX（単語単位の時刻つき文字起こし）
テロップ同期の精度上げ＋ピッチ解析用。初回はモデルのダウンロードで数分かかる。

In [ ]:
!pip -q install whisperx
import whisperx, json

device = "cuda"
model = whisperx.load_model("small", device, compute_type="float16", language="ja")
audio = whisperx.load_audio("/content/input.wav")
result = model.transcribe(audio, batch_size=16)
align_model, metadata = whisperx.load_align_model(language_code="ja", device=device)
result = whisperx.align(result["segments"], align_model, metadata, audio, device)

with open("/content/whisperx_result.json", "w", encoding="utf-8") as f:
    json.dump(result["segments"], f, ensure_ascii=False, indent=2)
print("OK: /content/whisperx_result.json")
files.download("/content/whisperx_result.json")

## ③ Real-ESRGAN（画像の4倍高解像度化）
512×896のような小さい静止画を 2048×3584 に拡大する。健全な静止画専用（ルール参照）。

In [ ]:
!git clone -q https://github.com/xinntao/Real-ESRGAN
%cd Real-ESRGAN
!pip -q install basicsr facexlib gfpgan
!pip -q install -r requirements.txt
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights
!python inference_realesrgan.py -n RealESRGAN_x4plus -i /content/input.png -o /content/results --outscale 4
print("OK: /content/results")

## ④ Qwen3-TTS（声クローン）— 次に試す用
なぎ本人の声を使ったクローン再演の検証用。枠だけ用意してある。

In [ ]:
# 検証時にここへ実装する
# !pip -q install qwen-tts などを想定
print("TODO")